# HPP Chapter 2 — Profiling to Find Bottlenecks
### Phase 01 · High Performance Python Ch 2 · Complete Reference Notebook

> *"Always be driven by the results of profiling. Profiling lets us find bottlenecks so we can do the least amount of work to get the biggest practical performance gain."* — High Performance Python

This notebook is a **standalone reference** for Chapter 2.
Read the actual chapter first. Return here to study, practice, and review.

**How to use:**
- Chapter content is covered faithfully and in order — Parts 1–10
- ML/DL extensions come **after** the chapter foundation — Parts 11–13
- Every profiling tool has a runnable example

---

**Sections**

| # | Chapter Foundation | # | ML / DL Extension |
|---|---|---|---|
| 1 | The Julia Set — why this example | 11 | Profiling a DataLoader — where's the bottleneck? |
| 2 | Simple timing — print, decorator, %timeit | 12 | memory_profiler on a training batch — OOM debugging |
| 3 | Unix `time` command | 13 | line_profiler on augmentation pipeline |
| 4 | `cProfile` — find the slow function | | |
| 5 | `snakeviz` — visualize cProfile output | | |
| 6 | `line_profiler` — find the slow line | | |
| 7 | `memory_profiler` — find the memory hog | | |
| 8 | `mprof` — memory over time | | |
| 9 | `py-spy` — profile running processes | | |
| 10 | `dis` module — read the bytecode | | |



---
# Part 1 — The Julia Set: Why This Example


## A CPU-bound problem with unpredictable complexity

The Julia set is an **embarrassingly parallel** fractal sequence — each pixel is calculated independently.
The chapter uses it because:

1. **Nonlinear behavior** — we can't predict how long each point will take
2. **CPU-bound with RAM pressure** — perfect for profiling both
3. **Deliberately suboptimal** — so we can identify slow statements

The core calculation:

```python
for z in coordinates:
    for iteration in range(maxiter):
        if abs(z) < 2.0:
            z = z*z + c
        else:
            break
```

**Key insight:** White regions in the output take longer to calculate (more iterations before escape).
This variance is what makes profiling results dramatic.


In [ ]:
# Complete Julia set implementation — the chapter's running example
import time
import math

# Constants for the complex plane region
x1, x2, y1, y2 = -1.8, 1.8, -1.8, 1.8
c_real, c_imag = -0.62772, -0.42193


def calc_pure_python(desired_width, max_iterations):
    """Create zs and cs lists, build Julia set, and time the calculation."""
    x_step = (x2 - x1) / desired_width
    y_step = (y1 - y2) / desired_width
    
    # Build coordinate lists
    x = []
    y = []
    ycoord = y2
    while ycoord > y1:
        y.append(ycoord)
        ycoord += y_step
    xcoord = x1
    while xcoord < x2:
        x.append(xcoord)
        xcoord += x_step
    
    # Build zs and cs lists (deliberately storing both for RAM profiling)
    zs = []
    cs = []
    for ycoord in y:
        for xcoord in x:
            zs.append(complex(xcoord, ycoord))
            cs.append(complex(c_real, c_imag))
    
    print(f"Length of x: {len(x)}")
    print(f"Total elements: {len(zs)}")
    
    start_time = time.time()
    output = calculate_z_serial_purepython(max_iterations, zs, cs)
    end_time = time.time()
    
    print(f"calculate_z_serial_purepython took {end_time - start_time:.3f} seconds")
    
    # Sanity check — deterministic output
    assert sum(output) == 33219980
    return output


def calculate_z_serial_purepython(maxiter, zs, cs):
    """CPU-bound calculation — this is what we'll profile."""
    output = [0] * len(zs)
    for i in range(len(zs)):
        n = 0
        z = zs[i]
        c = cs[i]
        while abs(z) < 2 and n < maxiter:
            z = z * z + c
            n += 1
        output[i] = n
    return output


if __name__ == "__main__":
    calc_pure_python(desired_width=1000, max_iterations=300)


---
# Part 2 — Simple Timing Approaches


## 2.1 — Print statements

The simplest approach. Quick and dirty, useful for first investigation.
But it quickly becomes unmanageable and clutters stdout.

## 2.2 — Timing decorator

Cleaner than print statements. Add `@timefn` to any function you care about.
The overhead is small but measurable if called millions of times.

## 2.3 — `%timeit` magic in IPython / Jupyter

The best tool for microbenchmarks. Runs the code many times and gives statistics.

**Important nuance from the chapter:**
- `timeit.py` (command line) uses the **minimum** value seen
- IPython/Jupyter `%timeit` switched in 2016 to use **mean and standard deviation**
- You can't compare results between the two — use one method consistently


In [ ]:
from functools import wraps

# ── Timing decorator ─────────────────────────────────────────────
def timefn(fn):
    @wraps(fn)
    def measure_time(*args, **kwargs):
        t1 = time.time()
        result = fn(*args, **kwargs)
        t2 = time.time()
        print(f"@timefn:{fn.__name__} took {t2 - t1:.6f} seconds")
        return result
    return measure_time


@timefn
def calculate_z_timed(maxiter, zs, cs):
    output = [0] * len(zs)
    for i in range(len(zs)):
        n = 0
        z = zs[i]
        c = cs[i]
        while abs(z) < 2 and n < maxiter:
            z = z * z + c
            n += 1
        output[i] = n
    return output


# ── Demonstrate the decorator ────────────────────────────────────
# Rebuild the lists (smaller grid for quick demo)
def get_test_data(width=500, maxiter=300):
    x_step = (x2 - x1) / width
    y_step = (y1 - y2) / width
    x = []
    y = []
    ycoord = y2
    while ycoord > y1:
        y.append(ycoord)
        ycoord += y_step
    xcoord = x1
    while xcoord < x2:
        x.append(xcoord)
        xcoord += x_step
    zs = []
    cs = []
    for ycoord in y:
        for xcoord in x:
            zs.append(complex(xcoord, ycoord))
            cs.append(complex(c_real, c_imag))
    return zs, cs


zs_small, cs_small = get_test_data(500)
print("Timing decorator:")
calculate_z_timed(300, zs_small, cs_small)

print("\n" + "─" * 50)
print("%timeit microbenchmark (run this cell multiple times to see variance):")
print("  %timeit calculate_z_serial_purepython(300, zs_small, cs_small)")
print("\n  Note: In Jupyter, %timeit shows mean ± std. deviation.")
print("  In terminal `python -m timeit`, shows the minimum.")


---
# Part 3 — The Unix `time` Command


## Measure at the OS level — outside Python

```bash
/usr/bin/time -p python julia1_nopil.py
```

The three results:

| Field | Meaning |
|-------|---------|
| `real` | Wall clock time (elapsed) |
| `user` | CPU time spent outside kernel functions |
| `sys` | CPU time spent in kernel functions |

**`user + sys`** = total CPU time. If this is much less than `real`, your program is I/O-bound or the system was busy.

**`Major page faults`** (from `--verbose`) indicate data was swapped to disk — a serious performance penalty.

```bash
/usr/bin/time --verbose python julia1_nopil.py
```

This works from any terminal. Not runnable inside the notebook, but the concept is critical:
**Before you profile inside Python, know whether you're CPU-bound or I/O-bound at the OS level.**



---
# Part 4 — `cProfile`: Find the Slow Function


## Built-in, hooks into CPython VM, measures every function call

**Overhead:** ~4 seconds on an 8-second function (50% slowdown).
But the information is worth it.

```bash
python -m cProfile -s cumulative julia1_nopil.py
```

The `-s cumulative` flag sorts by **cumulative time** — total time spent in the function including calls to subfunctions.

**Understanding the output (from the chapter's Example 2-8):**

| Column | Meaning |
|--------|---------|
| `ncalls` | Number of calls |
| `tottime` | Total time in function **excluding** subcalls |
| `percall` | `tottime / ncalls` |
| `cumtime` | Total time **including** subcalls |
| `percall` (second) | `cumtime / ncalls` |

**Key insight from the chapter:**
Inside `calculate_z_serial_purepython`, 34,219,980 calls to `abs()` take 3 seconds.
The per-call cost is negligible (`0.000`), but aggregated over 34M calls, it's the bottleneck.
This is why you must look at `tottime` (total), not `percall`.


In [ ]:
import cProfile
import pstats
import io

# ── Run cProfile programmatically ────────────────────────────────
def profile_function(func, *args, **kwargs):
    profiler = cProfile.Profile()
    profiler.enable()
    result = func(*args, **kwargs)
    profiler.disable()
    
    # Capture output
    stream = io.StringIO()
    stats = pstats.Stats(profiler, stream=stream)
    stats.sort_stats('cumulative')
    stats.print_stats(15)  # Show top 15 lines
    print(stream.getvalue())
    return result


print("=== cProfile output (top 15 lines, sorted by cumulative time) ===\n")
zs_big, cs_big = get_test_data(800)  # 640,000 elements — manageable
profile_function(calculate_z_serial_purepython, 300, zs_big, cs_big)

print("\n" + "─" * 60)
print("Interpretation:")
print("  - ncalls: number of times each function was called")
print("  - tottime: time INSIDE this function (excluding subcalls)")
print("  - cumtime: time including subcalls")
print("  - The {abs} line shows 34M calls to abs() — that's the real cost")


---
# Part 5 — `snakeviz`: Visualize `cProfile` Output


## Install: `pip install snakeviz`

```bash
python -m cProfile -o profile.stats julia1_nopil.py
snakeviz profile.stats
```

**What the visualization shows (Figure 2-5 in the chapter):**
- Entry point at the top of the diagram
- Each layer down is a function called from the function above
- **Width represents total execution time** — wider blocks = more time
- The unannotated block in the fifth layer (~25% of width) is the `abs()` function

This is the quickest way to communicate profiling results to others.
The diagram immediately shows where the time is spent without reading tables.

```python
# Code to generate stats file (run in terminal, not notebook)
import cProfile
cProfile.run('calc_pure_python(1000, 300)', 'profile.stats')
```

Then from terminal: `snakeviz profile.stats`



---
# Part 6 — `line_profiler`: Find the Slow Line


## Install: `pip install line_profiler`

`cProfile` tells you **which function** is slow. `line_profiler` tells you **which line**.

```bash
kernprof -l -v julia1_lineprofiler.py
```

**Flags:**
- `-l` : line-by-line (instead of function-level)
- `-v` : verbose output

**Key insight from the chapter's Example 2-6:**

```
Line #      Hits         Time  Per Hit   % Time  Line Contents
==============================================================
    17  34219980        0.5     38.0      while abs(z) < 2 and n < maxiter:
    18  33219980        0.5     30.8          z = z * z + c
    19  33219980        0.4     27.1          n += 1
```

**38% of time on the `while` test alone.** Even `n += 1` is expensive — Python's dynamic dispatch checks types every iteration.

The chapter then breaks the compound `while` into separate statements to measure each part:

```python
while True:
    not_yet_escaped = abs(z) < 2
    iterations_left = n < maxiter
    if not_yet_escaped and iterations_left:
        z = z * z + c
        n += 1
    else:
        break
```

This adds overhead but reveals that the `abs()` call is 2× more expensive than `n < maxiter`.


In [ ]:
# ── Decorated for line_profiler ─────────────────────────────────
# Run with: kernprof -l -v this_notebook.py (converted to .py first)
# Or use the magic command in Jupyter:
# %load_ext line_profiler
# %lprun -f calculate_z_serial_purepython calculate_z_serial_purepython(300, zs_big, cs_big)

print("=== How to run line_profiler ===\n")
print("In Jupyter:")
print("  %load_ext line_profiler")
print("  %lprun -f calculate_z_serial_purepython calculate_z_serial_purepython(300, zs_big, cs_big)")
print()
print("From terminal:")
print("  kernprof -l -v julia1_lineprofiler.py")
print()
print("What to look for:")
print("  - % Time column: lines with >10% are optimization targets")
print("  - Hits column: lines inside loops will have enormous hit counts")
print("  - The while test line will show ~38% time")


---
# Part 7 — `memory_profiler`: Find the Memory Hog


## Install: `pip install memory_profiler` (optional: `pip install psutil` for speed)

```bash
python -m memory_profiler julia1_memoryprofiler.py
```

**Key insight from the chapter's Example 2-10:**

```
Line #   Mem usage    Increment   Line Contents
==============================================
    12   133.973 MiB   7.609 MiB   output = [0] * len(zs)
    41   125.961 MiB   0.000 MiB   for ycoord in y:
    42   125.961 MiB   0.258 MiB       for xcoord in x:
    43   125.961 MiB   0.512 MiB           zs.append(complex(xcoord, ycoord))
```

**Interpretation:**
- The `output = [0] * len(zs)` line increased the process by ~7.6 MB
- The `zs` and `cs` lists increased memory from 48 MB to 125 MB (+77 MB)
- Not necessarily the exact size of the lists — just the amount the process grew

**Caveat from the chapter:** The `Increment` column can be buggy. Use the `Mem usage` column to track changes.

**The key question memory_profiler helps answer:**
> Could we use less RAM by rewriting this function? Or use more RAM and save CPU cycles by caching?


In [ ]:
# ── Decorated for memory_profiler ───────────────────────────────
# Run with: python -m memory_profiler this_file.py

print("=== How to run memory_profiler ===\n")
print("From terminal:")
print("  python -m memory_profiler julia1_memoryprofiler.py")
print()
print("In Jupyter:")
print("  %load_ext memory_profiler")
print("  %memit calculate_z_serial_purepython(300, zs_big, cs_big)  # single statement")
print("  %%memit  # for a block of code")
print("  # then run the cell with the code block")
print()
print("What to look for:")
print("  - Large Increment on list/dict creation lines")
print("  - Steady increase inside loops (temporary objects)")
print("  - The 'peak memory' line — where the OOM would happen")


---
# Part 8 — `mprof`: Memory Over Time


## Visualize memory usage as a function of time

```bash
mprof run julia1_memoryprofiler.py
mprof plot
```

**Key features from the chapter:**
- Samples by **time**, not by line — barely impacts runtime
- Shows when functions enter and exit (brackets in Figure 2-6)
- Labels can be added with context managers to annotate phases

**Example 2-11 from the chapter — using `profile.timestamp`:**

```python
@profile
def calculate_z_serial_purepython(maxiter, zs, cs):
    with profile.timestamp("create_output_list"):
        output = [0] * len(zs)
        time.sleep(1)
    with profile.timestamp("calculate_output"):
        for i in range(len(zs)):
            # ... calculation ...
    return output
```

The resulting plot (Figure 2-7) shows:
- `create_output_list` — brief spike at ~1.5 seconds
- `calculate_output` — slow linear increase from temporary objects
- A dashed vertical line (peak RAM usage) before program termination — garbage collection

**The optimization** — removing `zs` and `cs` lists (calculating coordinates on the fly) dropped RAM from 140 MB to 60 MB (Figure 2-8).


In [ ]:
print("=== How to use mprof ===\n")
print("From terminal:")
print("  mprof run python julia1_memoryprofiler.py")
print("  mprof plot")
print()
print("The plot shows:")
print("  - X-axis: time (seconds)")
print("  - Y-axis: memory usage (MiB)")
print("  - Brackets: function entry/exit")
print("  - Dashed line: peak RAM usage before garbage collection")


---
# Part 9 — `py-spy`: Profile Running Processes


## Install: `pip install py-spy`

**Key features:**
- Sampling profiler — almost no runtime impact
- Written in Rust
- Requires elevated privileges to introspect another process
- Works on Windows, Mac, Linux

```bash
# Find the PID of your running Python process
ps -A -o pid,rss,cmd | grep python

# Top-like live view
py-spy top --pid 15953

# Generate a flame chart
py-spy record -o profile.svg --pid 15953

# Run a script and profile it directly
py-spy record -o profile.svg -- python julia1_nopil.py
```

**What the flame chart shows (Figure 2-10):**
- Width = entire program runtime
- Each layer down = functions called from above
- Wider blocks = more time spent in that function

This is the tool for **production systems** — no code changes, minimal overhead, works on already-running processes.


In [ ]:
print("=== How to use py-spy ===\n")
print("1. Run your Python script in one terminal:")
print("   python julia1_nopil.py")
print()
print("2. In another terminal, find the PID:")
print("   ps -A -o pid,cmd | grep python")
print()
print("3. Profile the running process:")
print("   sudo py-spy top --pid <PID>")
print()
print("4. Generate a flame chart:")
print("   sudo py-spy record -o profile.svg --pid <PID>")
print()
print("Or profile a script directly:")
print("   py-spy record -o profile.svg -- python julia1_nopil.py")


---
# Part 10 — The `dis` Module: Read the Bytecode


## PNP Ch 57 — Understanding CPython's stack-based VM

```python
import dis
dis.dis(calculate_z_serial_purepython)
```

**What the columns mean:**

| Column | Meaning |
|--------|---------|
| First | Line number in original file |
| `>>` | Jump destination |
| Third | Operation address |
| Fourth | Operation name (opcode) |
| Fifth | Parameter for the operation |
| Sixth | Annotation (original Python) |

**Example from the chapter (Example 2-14):**

```
11          0 LOAD_CONST           1 (0)
            2 BUILD_LIST           1
            4 LOAD_GLOBAL          0 (len)
            6 LOAD_FAST            1 (zs)
            8 CALL_FUNCTION        1
           10 BINARY_MULTIPLY
           12 STORE_FAST           3 (output)
```

**Key insight:** More bytecode instructions ≠ always slower, but inside a tight loop, each instruction matters.
The example comparing `fn_expensive` (17 bytecode lines) vs `fn_terse` (6 lines) shows a 2.9× speedup.

**The rule of thumb (must be profiled, not assumed):**
Built-in functions (`sum`, `map`, `any`) often generate less bytecode and run in C → faster.
But profile to confirm — microbenchmarks can mislead.


In [ ]:
import dis

def fn_expensive(upper=1_000_000):
    total = 0
    for n in range(upper):
        total += n
    return total

def fn_terse(upper=1_000_000):
    return sum(range(upper))

print("=== fn_expensive bytecode ===")
dis.dis(fn_expensive)
print("\n=== fn_terse bytecode ===")
dis.dis(fn_terse)
print("\nfn_expensive: 17 bytecode instructions")
print("fn_terse:      6 bytecode instructions + sum() runs in C")
print("\nSpeed difference from the chapter: fn_terse is 2.9× faster")


---
# Part 11 — ML Extension: Profiling a DataLoader


## Where is the bottleneck? DataLoader or model?

A common ML problem: training is slow. Is it the DataLoader (I/O-bound) or the model forward/backward (CPU/GPU-bound)?

**The profiling strategy from the chapter:**
1. Start with `cProfile` on a single training epoch
2. Look at `cumtime` — what's taking the most total time?
3. Drill into the top function with `line_profiler`
4. Check memory with `memory_profiler` if OOM occurs

**Common bottlenecks in ML data pipelines:**

| Symptom | Likely cause | Profiler to use |
|---------|--------------|-----------------|
| High CPU, GPU idle | Python transforms in DataLoader | `cProfile`, `line_profiler` |
| Low CPU, GPU idle, high memory | I/O wait (disk too slow) | `py-spy` (see I/O wait in flame chart) |
| Memory grows over time | Memory leak in augmentation | `memory_profiler` mprof plot |
| Sudden OOM | Batch size too large or memory spike | `memory_profiler` with `--pdb-mem` |


In [ ]:
import numpy as np
from typing import List, Tuple
import time

# ── Simulate a DataLoader with expensive transforms ─────────────
class SlowDataset:
    def __init__(self, size: int, feature_dim: int):
        self.data = np.random.randn(size, feature_dim).astype(np.float32)
        self.labels = np.random.randint(0, 10, size=size)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Deliberately slow: creating new objects, expensive ops
        x = self.data[idx]
        y = self.labels[idx]
        
        # Expensive normalization (should be precomputed)
        x = (x - x.mean()) / (x.std() + 1e-8)
        
        # Expensive augmentation
        noise = np.random.randn(*x.shape) * 0.1
        x = x + noise
        
        return x, y


class FastDataset:
    def __init__(self, size: int, feature_dim: int):
        # Pre-normalized data
        raw = np.random.randn(size, feature_dim).astype(np.float32)
        mean = raw.mean(axis=0, keepdims=True)
        std = raw.std(axis=0, keepdims=True) + 1e-8
        self.data = (raw - mean) / std
        self.labels = np.random.randint(0, 10, size=size)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # No per-sample normalization — just return
        return self.data[idx], self.labels[idx]


def train_epoch(dataset, batch_size=32):
    """Simulate one training epoch."""
    indices = np.random.permutation(len(dataset))
    total_loss = 0.0
    
    for start in range(0, len(dataset), batch_size):
        batch_indices = indices[start:start + batch_size]
        
        # Data loading (the part we're profiling)
        batch_x = []
        batch_y = []
        for idx in batch_indices:
            x, y = dataset[idx]
            batch_x.append(x)
            batch_y.append(y)
        
        # Simulate forward pass
        batch_x = np.stack(batch_x)
        batch_y = np.array(batch_y)
        
        # Dummy loss computation
        loss = np.sum(batch_x ** 2) + np.sum(batch_y)
        total_loss += loss
    
    return total_loss


# ── Benchmark ────────────────────────────────────────────────────
size = 5000
dim = 128

slow_ds = SlowDataset(size, dim)
fast_ds = FastDataset(size, dim)

t0 = time.perf_counter()
train_epoch(slow_ds)
t_slow = time.perf_counter() - t0

t0 = time.perf_counter()
train_epoch(fast_ds)
t_fast = time.perf_counter() - t0

print(f"Slow DataLoader (per-sample normalization + noise): {t_slow:.2f}s")
print(f"Fast DataLoader (pre-normalized data):               {t_fast:.2f}s")
print(f"Speedup: {t_slow / t_fast:.1f}×")
print()
print("Profiling lesson: cProfile would show __getitem__ as the bottleneck.")
print("line_profiler would show the exact line: x = (x - x.mean()) / (x.std() + 1e-8)")
print("The fix: pre-compute normalization, don't do it per-sample.")


---
# Part 12 — ML Extension: `memory_profiler` on a Training Batch


## Debugging OOM (Out of Memory) in training

The chapter's `--pdb-mem` flag is invaluable for OOM debugging:

```bash
python -m memory_profiler --pdb-mem=1024 train.py
```

This drops you into `pdb` when memory exceeds 1024 MB, right at the line that triggered the allocation.

**Common ML memory patterns to profile:**

1. **Batch collation** — stacking variable-length sequences creates copies
2. **Data augmentation** — generating augmented images on the fly doubles memory
3. **Gradient accumulation** — storing intermediate activations
4. **Checkpointing** — saving model states at every epoch

**The chapter's `mprof plot` technique applied to training:**

```python
@profile
def train_step(batch):
    with profile.timestamp("data_to_gpu"):
        batch = batch.cuda()
    with profile.timestamp("forward"):
        logits = model(batch)
    with profile.timestamp("backward"):
        loss.backward()
    with profile.timestamp("optimizer"):
        optimizer.step()
```

The resulting plot shows which phase of training consumes memory.


In [ ]:
print("=== Debugging OOM with memory_profiler ===\n")
print("1. Run with the --pdb-mem flag:")
print("   python -m memory_profiler --pdb-mem=2048 train.py")
print()
print("2. When memory exceeds 2048 MB, you drop into pdb at the offending line:")
print("   > /path/to/train.py(42)train_step()")
print("   -> batch = batch.cuda()  # ← memory spiked here")
print("   (Pdb) print(batch.shape)  # inspect")
print("   (Pdb) print(torch.cuda.memory_allocated() / 1024**3)  # GB used")
print()
print("3. Use mprof plot to see memory over time:")
print("   mprof run python train.py")
print("   mprof plot")
print()
print("The plot will show which training phase (data loading, forward, backward, optimizer)")
print("causes the memory spike.")


---
# Part 13 — ML Extension: `line_profiler` on Augmentation Pipeline


## Finding the slow transform

Image augmentation is often the hidden bottleneck in computer vision pipelines.

```python
@profile
def transform(image):
    image = random_rotate(image)      # line 1
    image = random_crop(image, 224)   # line 2
    image = random_flip(image)        # line 3
    image = normalize(image)          # line 4
    return image
```

Run:
```bash
kernprof -l -v train.py
```

The output will show % Time per line. If line 2 (random_crop) is 70%, that's your target.

**The chapter's hypothesis-testing method applied to augmentation:**

1. **Hypothesis:** `random_crop` is the slowest transform because it requires memory allocation
2. **Test:** Comment out `random_crop`, re-run `line_profiler`
3. **Conclusion:** If time drops by 70%, hypothesis confirmed
4. **Optimization:** Pre-crop? Use `torchvision`'s C implementation? Move to GPU?

**The rule from the chapter:** Never test two things at once. Change one transform at a time and re-profile.


In [ ]:
print("=== Profiling an augmentation pipeline ===\n")
print("1. Add @profile to your transform function")
print()
print("2. Run: kernprof -l -v train.py")
print()
print("3. Interpret the output:")
print("   Line #     Hits     Time    % Time   Line Contents")
print("   ==================================================")
print("      10   100000    0.5s     12.0%     image = random_rotate(image)")
print("      11   100000    3.2s     70.0%     image = random_crop(image, 224)  ← BOTTLENECK")
print("      12   100000    0.4s      8.0%     image = random_flip(image)")
print("      13   100000    0.5s     10.0%     image = normalize(image)")
print()
print("4. Hypothesis: random_crop is slow because it allocates new memory per sample")
print("5. Fix: Move to torchvision's C implementation or use multiprocessing")
print("6. Re-profile to confirm the fix worked")


---
# Summary — HPP Ch 2: One-Page Reference


## The single rule

> **Profile before you optimize. Always be driven by the results of profiling.**
> Form a hypothesis, test it, measure the change. Never guess.

---

## Profiling tools quick reference

| Tool | What it tells you | Command | Best for |
|------|-------------------|---------|----------|
| `%timeit` | Execution time of a single statement | `%timeit fn()` | Microbenchmarks |
| `time` (Unix) | CPU vs real time, page faults | `/usr/bin/time -v python script.py` | OS-level check |
| `cProfile` | Which function is slow | `python -m cProfile -s cumulative script.py` | First pass |
| `snakeviz` | Visualize cProfile output | `snakeviz profile.stats` | Team communication |
| `line_profiler` | Which line is slow | `kernprof -l -v script.py` | After cProfile |
| `memory_profiler` | Which line allocates memory | `python -m memory_profiler script.py` | RAM bottlenecks |
| `mprof` | Memory over time | `mprof run script.py; mprof plot` | Memory spikes |
| `py-spy` | Profile running process | `py-spy top --pid <PID>` | Production systems |
| `dis` | Bytecode instructions | `dis.dis(function)` | Understanding VM |

---

## The chapter's workflow

```
1. Unix `time` → Is it CPU-bound or I/O-bound?
       ↓
2. `cProfile` → Which function?
       ↓
3. `line_profiler` → Which line?
       ↓
4. Optimize one thing
       ↓
5. Run unit tests (with no-op @profile decorator)
       ↓
6. Re-profile to confirm
```

---

## Key insights from the chapter

| Insight | Why it matters |
|---------|----------------|
| `tottime` > `percall` | Look at total time, not per-call — 34M calls to abs() add up |
| `cProfile` adds overhead | 8s → 12s (50% slowdown). Worth it for the data |
| `line_profiler` adds more | 8s → 49s (6× slower). Use on small representative data |
| `%timeit` in Jupyter | Uses mean ± std dev. `timeit.py` uses min. Don't compare across them |
| Breaking compound statements | Adds overhead but reveals true cost of each part |
| Disable Turbo Boost | CPU frequency scaling skews results |
| Hypothesis first | Write down what you expect before profiling |

---

## ML connection map

| Profiling tool | ML use case |
|----------------|-------------|
| `cProfile` | Find slow function in training loop (DataLoader vs model) |
| `line_profiler` | Find slow transform in augmentation pipeline |
| `memory_profiler` | Debug OOM during batch collation |
| `mprof` | See memory growth over epoch — memory leak detection |
| `py-spy` | Profile a long-running training job without stopping it |
| `--pdb-mem` | Drop into debugger exactly when OOM occurs |

---

## Unit testing during optimization

```python
# No-op decorator so @profile doesn't break tests
if 'line_profiler' not in dir() and 'profile' not in dir():
    def profile(func):
        return func
```

Without this, your unit tests fail with `NameError: name 'profile' is not defined`.
With it, the same code runs under `pytest`, `kernprof`, and `python -m memory_profiler`.

---

## The Julia set lesson

The chapter's running example teaches one thing above all:
**Start with a slow, correct implementation. Profile. Find the 38% line. Fix it.**
Don't start with an optimized version — you won't know what you optimized for.
